# B · Compare host feature profiles by pathogen group

This notebook compares the **resolved question sets** for every registered host and pathogen group. Each pathogen group gets a square host × host matrix. A cell is the Jaccard similarity of the two hosts’ applicable feature sets:

\[
J(A,B) = rac{|A \cap B|}{|A \cup B|}
\]

A value of 1 means the resolved profiles contain the same features; 0 means they share none. Feature identity includes `feature_id`, `question_version`, and `scope`, so a changed question version is not silently treated as identical. Host and pathogen applicability is resolved with the same `resolve_profile` function used by the V4 workflow.

By default, the matrix includes all resolved scopes. Set `SCOPE_FILTER = {"pathogen_host_pair"}` below to compare only host-pair questions; this avoids pathogen-level traits that are shared across hosts by design from inflating the similarity. A zero-union comparison is left undefined (`NaN`).


In [ ]:
import sys
from itertools import combinations
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "pathogen_atlas_v4").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "pathogen_atlas_v4").is_dir():
    raise RuntimeError("Open this notebook from the Pathogen Atlas V4 repository")
sys.path.insert(0, str(PROJECT_ROOT))

from pathogen_atlas_v4.features import resolve_profile
from pathogen_atlas_v4.host_taxonomy.ncbi import PILOT_HOSTS

HOSTS_PATH = PROJECT_ROOT / "configs/host_choices.yaml"
PATHOGENS_PATH = PROJECT_ROOT / "configs/pathogen_taxonomies.yaml"
FEATURES_PATH = PROJECT_ROOT / "configs/features.yaml"
with HOSTS_PATH.open(encoding="utf-8") as stream:
    host_registry = yaml.safe_load(stream)
with PATHOGENS_PATH.open(encoding="utf-8") as stream:
    pathogen_registry = yaml.safe_load(stream)
with FEATURES_PATH.open(encoding="utf-8") as stream:
    feature_catalogue = yaml.safe_load(stream)

RELATIONSHIP_CONTEXT = "disease"
PROFILE_VERSION = str(feature_catalogue.get("profile_version", ""))
# None includes every resolved scope; use {"pathogen_host_pair"} for host-specific questions only.
SCOPE_FILTER = None


## 1 · Resolve each host × pathogen-group question set

Host aliases are collapsed by NCBI taxon ID, matching the question-generation notebook. The resolver applies curated host-group traits, host-taxon traits, pathogen-group traits, and the disease relationship context before collecting feature IDs.


In [ ]:
# Deduplicate host aliases while keeping the first configured display name.
host_aliases_by_id = {}
for name, taxon_id in host_registry["host_name_to_id"].items():
    host_aliases_by_id.setdefault(str(taxon_id), []).append(str(name))

hosts = []
for host_id, aliases in host_aliases_by_id.items():
    taxon_id = host_id.removeprefix("ncbi:")
    hosts.append({
        "host_id": host_id,
        "taxon_id": taxon_id,
        "label": aliases[0],
        "aliases": aliases[1:],
        "host_group": PILOT_HOSTS.get(taxon_id, "other"),
    })
pathogen_groups = [
    {"pathogen_group": str(group_id), "label": str(group.get("label", group_id))}
    for group_id, group in pathogen_registry["pathogen_groups"].items()
]

feature_sets = {}
feature_records = {}
profile_rows = []
for group in pathogen_groups:
    group_id = group["pathogen_group"]
    for host in hosts:
        resolved = resolve_profile(
            feature_catalogue,
            pathogen_group=group_id,
            host_group=host["host_group"],
            host_taxon_id=host["taxon_id"],
            relationship_context=RELATIONSHIP_CONTEXT,
            profile_version=PROFILE_VERSION,
        )
        records = resolved["ordered_features"]
        if SCOPE_FILTER is not None:
            records = [feature for feature in records if feature.get("scope") in SCOPE_FILTER]
        # Include version and scope in identity to avoid equating incompatible definitions.
        identities = {
            (str(feature["feature_id"]), str(feature.get("question_version", "")), str(feature["scope"]))
            for feature in records
        }
        feature_sets[(group_id, host["host_id"])] = identities
        feature_records[(group_id, host["host_id"])] = records
        profile_rows.append({
            "pathogen_group": group_id,
            "host": host["label"],
            "host_id": host["host_id"],
            "host_group": host["host_group"],
            "feature_count": len(identities),
            "profile_hash": resolved["resolved_profile_hash"],
        })

profile_counts = pd.DataFrame(profile_rows)
print(f"Resolved {len(pathogen_groups)} pathogen groups × {len(hosts)} distinct hosts.")
display(profile_counts.pivot(index="host", columns="pathogen_group", values="feature_count").reindex(
    index=[host["label"] for host in hosts], columns=[group["pathogen_group"] for group in pathogen_groups]
))


## 2 · Build the Jaccard matrices

Rows and columns use the same configured host order. The intersection and union counts are retained in a pairwise table so each matrix value can be inspected as `shared features / features in either profile`.


In [ ]:
host_labels = [host["label"] for host in hosts]
host_id_by_label = {host["label"]: host["host_id"] for host in hosts}
matrices = {}
pairwise_rows = []

for group in pathogen_groups:
    group_id = group["pathogen_group"]
    matrix = pd.DataFrame(np.nan, index=host_labels, columns=host_labels, dtype=float)
    for left, right in combinations(hosts, 2):
        left_set = feature_sets[(group_id, left["host_id"])]
        right_set = feature_sets[(group_id, right["host_id"])]
        intersection = left_set & right_set
        union = left_set | right_set
        score = len(intersection) / len(union) if union else np.nan
        matrix.loc[left["label"], right["label"]] = score
        matrix.loc[right["label"], left["label"]] = score
        pairwise_rows.append({
            "pathogen_group": group_id,
            "host_a": left["label"],
            "host_b": right["label"],
            "features_a": len(left_set),
            "features_b": len(right_set),
            "intersection": len(intersection),
            "union": len(union),
            "jaccard": score,
            "shared_feature_ids": "; ".join(sorted(item[0] for item in intersection)),
        })
    for host in hosts:
        own_set = feature_sets[(group_id, host["host_id"])]
        matrix.loc[host["label"], host["label"]] = 1.0 if own_set else np.nan
    matrices[group_id] = matrix

pairwise_comparisons = pd.DataFrame(pairwise_rows)


## 3 · Visualize one square matrix per pathogen group

The common 0–1 color scale makes values comparable between panels. Each cell is annotated with its Jaccard fraction; diagonal cells are 1 for nonempty profiles.


In [ ]:
for group in pathogen_groups:
    group_id = group["pathogen_group"]
    matrix = matrices[group_id]
    fig, ax = plt.subplots(figsize=(10, 9))
    image = ax.imshow(matrix.to_numpy(dtype=float), vmin=0, vmax=1, cmap="YlGnBu")
    ax.set_xticks(range(len(host_labels)), host_labels, rotation=45, ha="right")
    ax.set_yticks(range(len(host_labels)), host_labels)
    ax.set_title(f"{group['label']} · host feature-set similarity (Jaccard)", pad=14)
    ax.set_xlabel("Host")
    ax.set_ylabel("Host")
    ax.set_aspect("equal")
    for row in range(len(host_labels)):
        for col in range(len(host_labels)):
            value = matrix.iat[row, col]
            if pd.notna(value):
                ax.text(col, row, f"{value:.2f}", ha="center", va="center",
                        color="white" if value >= 0.58 else "#183247", fontsize=8)
    fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04, label="Jaccard similarity")
    fig.tight_layout()
    plt.show()


## 4 · Inspect pairwise overlaps

Use this table to see the numerator and denominator behind each off-diagonal cell, plus the shared IDs. Sort by pathogen group or Jaccard value to find the most similar and most different host profiles.


In [ ]:
display(
    pairwise_comparisons.sort_values(
        ["pathogen_group", "jaccard", "host_a", "host_b"],
        ascending=[True, False, True, True],
        na_position="last",
    ).reset_index(drop=True)
)


## Interpretation notes

- Jaccard compares **which questions apply**, not their answers or the amount of evidence collected.
- A shared feature means the same `feature_id`, `question_version`, and `scope` resolved for both hosts.
- `pathogen`-scope features may apply identically across hosts; use `SCOPE_FILTER = {"pathogen_host_pair"}` if the question is specifically how host-dependent the profiles are.
- A value of 0 means the two resolved sets have no feature identities in common. An empty union is undefined, not zero.
- The host trait assignments and feature applicability rules are curated inputs. A missing host trait can exclude a question from that host’s set.
